# T1. 팀원용 실험 노트북 — 내 Colab, 내 구글 드라이브에서 돌리고 결과만 보낸다

**흐름**: ① 내 드라이브에 코드·데이터 올리기 → ② 내 이름과 **내 모델** 정하기(방법 A: 설정만, 방법 B: 직접 짜기) → ③ "모두 실행" → ④ 끝나면 `ner/to_send/` 에 생긴 **zip 파일 하나를 최세진에게 보내기**. 비교·그림은 모아서 한 번에 한다.

**처음 한 번만: 내 구글 드라이브에 이렇게 놓기** (파일은 Notion "평가 기준과 결과 저장 방법" 페이지에서 내려받기)
```
내 드라이브/
└─ ner/
   ├─ src/                    ← ner_code.zip 을 풀면 나오는 src 폴더 그대로
   └─ data/split_v3/
      ├─ train_part1.jsonl    ← train_300k_part1~3.zip 을 풀어서 나온 3개 (이어 붙이기는 노트북이 함)
      ├─ train_part2.jsonl
      ├─ train_part3.jsonl
      ├─ dev.jsonl            ← dev.zip
      └─ test.jsonl           ← test.zip
```
**GPU**: 런타임 → 런타임 유형 변경 → A100(가장 빠름) 또는 L4. 30만 문장 3에폭 기준 A100 약 50분, L4 약 75분(기본 헤드). CRF는 약 2배.

**규칙**: 학습 데이터(30만)·dev·test·채점기는 모두 같게 쓴다. 모델과 학습 설정은 각자 자유롭게 정하되, 바꾼 내용을 실험 이름과 결과 파일에 남긴다. 그래야 모든 팀원 결과를 한 표에서 비교할 수 있다.

**최종 가중치(10/23부터 항상 저장)**: 방법 A는 학습이 끝나면 예측·채점보다 먼저 `내 드라이브/ner_models/{실험}/` 에 가중치와 토크나이저를 저장하고 다시 읽어 확인한다(실패하면 멈춤). 방법 B는 `save_custom_weights(...)` 를 꼭 부른다. 6번 셀은 가중치가 하나라도 없으면 런타임을 반납하지 않는다.

In [ ]:
!pip -q install -U transformers datasets accelerate pytorch-crf kiwipiepy
import os, sys, json, glob, torch
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/ner"                 # 내 드라이브의 ner 폴더
SPLIT   = f"{PROJECT}/data/split_v3"
RESULTS = f"{PROJECT}/reports/results"
CKPT    = "/content/drive/MyDrive/ner_ckpt"             # 끊겨도 마지막 에폭부터 이어서 학습
sys.path.insert(0, f"{PROJECT}/src")
import importlib, ner_lib; importlib.reload(ner_lib)
MODELS = "/content/drive/MyDrive/ner_models"   # 최종 가중치(실험마다 폴더). 학습 직후 저장하고 다시 읽어 확인한다. 지우지 말 것
ner_lib.DEFAULTS.update(model_dir=MODELS)
from ner_lib import run_experiment, load_results, results_table
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형에서 GPU 선택!)")

## 1. 파일 확인 (학습 세 조각이 있으면 자동으로 이어 붙임)

In [ ]:
os.makedirs(SPLIT, exist_ok=True)
tr = f"{SPLIT}/train.jsonl"; parts = [f"{SPLIT}/train_part{i}.jsonl" for i in (1, 2, 3)]
if not os.path.exists(tr) and all(os.path.exists(p) for p in parts):
    with open(tr, "w", encoding="utf-8") as fo:
        for p in parts:
            with open(p, encoding="utf-8") as fi: fo.write(fi.read())
    print("세 조각을 이어 붙여 train.jsonl 을 만들었습니다")
need = {"train.jsonl": 300000, "dev.jsonl": 10000, "test.jsonl": 10000}
for f, n in need.items():
    p = f"{SPLIT}/{f}"; got = sum(1 for _ in open(p, encoding="utf-8")) if os.path.exists(p) else 0
    print(f"{f:12s} {got:>7,}줄 {'OK' if got == n else '← 기준 ' + format(n, ',') + '줄과 다름!'}")
    assert got == n, f"{f} 줄 수가 다릅니다. 위 폴더 구조대로 올렸는지 확인하세요."
assert os.path.exists(f"{PROJECT}/src/ner_lib.py") and os.path.exists(f"{PROJECT}/src/evaluate.py"), "src 폴더가 없습니다"

## 2. 내 모델 정하기 — 방법 A: 우리 학습 코드 + 내 설정

**각자 자기 모델을 정해서 돌린다.** 방법은 둘 중 하나(둘 다 해도 된다).
- **방법 A (이 절)**: 우리 공용 학습 코드를 그대로 쓰고 **모델과 설정만 내가 정한다.** 학습·채점·오답 저장·에폭별 기록·끊겨도 이어서 하기가 자동이다.
- **방법 B (3-1절)**: 모델 구조와 학습 코드를 **직접 짠다.** 예측 함수만 약속된 형식으로 만들면 채점·저장은 같다.

방법 A에서 `MY_EXPS` 에 정할 수 있는 것

| 항목 | 값 | 기본값 |
|---|---|---|
| `model` | Hugging Face 모델 이름 (한국어 지원 인코더면 무엇이든) | 필수 |
| `head` | `"softmax"`(토큰마다 따로 분류) / `"crf"`(꼬리표 순서 규칙까지 학습) | softmax |
| `loss` | `"ce"`(일반) / `"weighted"`(드문 태그를 틀리면 더 크게 벌점) | ce |
| `pretok` | `None`(원문) / `"morph"`(형태소 경계에 공백을 넣어 입력) | None |
| `lr`, `epochs`, `batch`, `grad_accum` | 학습률, 에폭, 배치 | 5e-5, 3, 16, 1 |

- 실험 이름은 **자동으로** `T_{내이름}_{키}_300k` 가 된다. 키에 무엇을 바꿨는지 드러나게 짓는다(예: `xlmr`, `bert_crf`, `roberta_lr3e5`).
- 비교 기준(팀 공통 기준선, klue/roberta-base·30만·기본 설정): dev F1 0.7466, test F1 0.7306. 이보다 나은지, 무엇을 다르게 맞히는지를 본다.
- 학습률·에폭 같은 설정은 **dev 점수로만** 고르고, 데이터·dev·test는 바꾸지 않는다.
- 방법 B만 할 거면 `MY_EXPS` 를 비워 두면 된다.

In [ ]:
MEMBER = "hong"          # ← 내 이름 (영어)

MY_EXPS = {
    # "키": dict(model="Hugging Face 이름", head=..., loss=..., pretok=..., lr=..., epochs=...),
    # 예시 (주석을 풀어 쓰거나 내 것으로 바꾸기)
    # "xlmr":       dict(model="xlm-roberta-base"),
    # "bert_crf":   dict(model="klue/bert-base", head="crf"),
    # "electra_w":  dict(model="monologg/koelectra-base-v3-discriminator", loss="weighted"),
}

assert MEMBER not in ("", "hong"), "MEMBER 를 내 이름(영어)으로 바꾸세요"
COMMON = dict(train=f"{SPLIT}/train.jsonl", dev=f"{SPLIT}/dev.jsonl", test=f"{SPLIT}/test.jsonl", out_dir=RESULTS,
              ckpt_dir=CKPT, member=MEMBER, stage=f"team_{MEMBER}", max_len=512, lr=5e-5, epochs=3, batch=16, seed=42)
EXPS = {k: {**v, "exp": f"T_{MEMBER}_{k}_300k"} for k, v in MY_EXPS.items()}
for k, v in EXPS.items(): assert "model" in v, f"{k}: model 을 정하세요"
print("방법 A로 돌릴 실험:", [v["exp"] for v in EXPS.values()] or "없음 (방법 B만)")

## 3. 실행 (방법 A)
연결이 끊기면 0번 셀부터 다시 "모두 실행". 끝난 실험은 건너뛰고, 진행 중이던 실험은 마지막 에폭부터 이어서 학습한다. GPU별 시간(30만·3에폭, 기본 헤드): A100 약 50분, L4 약 75분. CRF는 약 2배, large 모델은 약 3배.

In [ ]:
for k, cfg in EXPS.items():
    run_experiment({**COMMON, **cfg})

## 3-1. 방법 B — 내가 직접 짠 모델 쓰기

**학습 코드를 직접 작성하기 (구조·학습 방식 자유)**: 아래 셀에서 모델을 만들고 학습한 뒤, **문장 리스트를 받아 문장마다 개체 목록 `[[시작, 끝, 태그], ...]` 을 돌려주는 함수** `my_predict` 만 만든다. 위치는 원문 글자 위치(끝은 포함 안 함)다. 토큰 분류 모델이면 `ner_lib.decode_spans(꼬리표들, offset_mapping)` 으로 바꿀 수 있다. LLM·규칙·사전 방식도 이 함수만 맞추면 된다.

`score_custom` 이 dev·test 전체를 예측해 **공용 채점기로 채점하고, 예측·점수·오답을 다른 실험과 같은 형식으로 저장**한다. 그러면 5번 셀이 함께 묶어 보낸다.

In [ ]:
import evaluate as ev, time
CUSTOM_EXPS = []        # score_custom 으로 만든 실험 이름이 자동으로 들어감 (5번 셀에서 함께 묶음)

def score_custom(exp, predict_fn, note=""):
    # predict_fn(list[str]) -> list[list[[s, e, tag]]] 을 dev·test 전체에 돌려 공용 기준으로 채점·저장
    out = {}
    for sp in ("dev", "test"):
        rows = ner_lib.load_jsonl(f"{SPLIT}/{sp}.jsonl"); t0 = time.time()
        preds = predict_fn([r["text"] for r in rows]); assert len(preds) == len(rows), "예측 개수가 문장 수와 다릅니다"
        ner_lib.save_preds(f"{RESULTS}/preds/{exp}_{sp}.jsonl", rows, [[tuple(x) for x in p] for p in preds])
        out[sp] = ev.score(rows, ev.load_pred(f"{RESULTS}/preds/{exp}_{sp}.jsonl"), exp, sp, RESULTS,
                           {"member": MEMBER, "stage": "custom", "note": note, "infer_sec": round(time.time() - t0, 1)})
        print(f"{exp} [{sp}] F1 {out[sp]['metrics']['ALL']['f1']} macro {out[sp]['metrics']['MACRO_F1']}")
    if exp not in CUSTOM_EXPS: CUSTOM_EXPS.append(exp)
    return out

def save_custom_weights(exp, model, tokenizer=None):
    # 방법 B로 직접 학습한 모델의 가중치를 내 드라이브/ner_models/{exp} 에 저장 (꼭 부를 것)
    d = f"{MODELS}/{exp}"; os.makedirs(d, exist_ok=True)
    if hasattr(model, "save_pretrained"): model.save_pretrained(d)
    else: torch.save(model.state_dict(), f"{d}/model_state.pt")
    if tokenizer is not None: tokenizer.save_pretrained(d)
    ner_lib.SESSION_EXPS.append((exp, d)); print("가중치 저장:", d, os.listdir(d))

# 직접 학습한 모델이면: save_custom_weights("M9_내이름_모델_300k", model, tokenizer)  ← score_custom 전에 부르기

# ── 예시: 학습 데이터에서 본 개체를 그대로 찾는 '사전 매칭' 모델 (내 모델로 바꿔 쓰기) ──
# train = ner_lib.load_jsonl(f"{SPLIT}/train.jsonl")
# vocab = {}
# for r in train:
#     for s, e, t in r["spans"]: vocab.setdefault(r["text"][s:e], t)
# words = sorted((w for w in vocab if len(w) >= 2), key=len, reverse=True)[:20000]
# def my_predict(texts):
#     out = []
#     for t in texts:
#         spans, used = [], [False] * len(t)
#         for w in words:
#             i = t.find(w)
#             while i >= 0:
#                 if not any(used[i:i + len(w)]): spans.append([i, i + len(w), vocab[w]]); used[i:i + len(w)] = [True] * len(w)
#                 i = t.find(w, i + 1)
#         out.append(sorted(spans))
#     return out
# score_custom(f"T_{MEMBER}_dict-match_300k", my_predict, note="학습셋 사전 정확 매칭")

## 4. 내 결과 확인

In [ ]:
mine = [r for r in load_results(RESULTS) if r["exp"] in {v["exp"] for v in EXPS.values()}]
results_table(mine)[["exp", "model", "head", "loss", "dev_F1", "test_F1", "test_P", "test_R", "macro_F1", "train_min"]]

## 5. 보낼 파일 만들기 → 이 zip 하나만 최세진에게 보낸다
실험마다 아래 파일을 묶는다. **zip 파일 이름에 내 이름과 시각이 들어간다.** 드라이브의 `ner/to_send/` 에서 공유 링크를 만들거나 내려받아 보내면 된다.

| 파일 | 내용 | 왜 필요한가 |
|---|---|---|
| `{실험}.json` | 설정, 에폭별 dev 점수, 학습 시간 | 에폭 곡선·시간 비교 |
| `scores/{실험}_dev.json`, `_test.json` | 공용 채점기 점수(태그·카테고리·오답 유형 수) | 점수 비교표 |
| `preds/{실험}_dev.jsonl`, `_test.jsonl` | 예측 원본 | **앙상블·오답 보완 분석** |
| `errors/{실험}_dev.jsonl`, `_test.jsonl` | 오답 목록 | 오답 분석 |
| `runs.jsonl` | 실행 기록 | 기록 |

**가중치는 zip에 넣지 않는다**(실험마다 약 440MB). 내 드라이브 `ner_models/{실험}/` 에 남아 있으니 지우지 말고, 2단계(금융 태그) 학습에 쓸 모델로 뽑히면 그 폴더를 공유한다.

In [ ]:
import zipfile, time
send_dir = f"{PROJECT}/to_send"; os.makedirs(send_dir, exist_ok=True)
zp = f"{send_dir}/results_{MEMBER}_{time.strftime('%m%d_%H%M')}.zip"
files, missing = [], []
for e in [v["exp"] for v in EXPS.values()] + list(globals().get("CUSTOM_EXPS", [])):
    for rel in [f"{e}.json"] + [f"{d}/{e}_{sp}.{x}" for d, x in (("scores", "json"), ("preds", "jsonl"), ("errors", "jsonl")) for sp in ("dev", "test")]:
        if os.path.exists(f"{RESULTS}/{rel}"): files.append(rel)
        elif not (rel == f"{e}.json" and e in globals().get("CUSTOM_EXPS", [])): missing.append(rel)   # 직접 만든 모델은 {실험}.json 없음
if os.path.exists(f"{RESULTS}/runs.jsonl"): files.append("runs.jsonl")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in files: z.write(f"{RESULTS}/{rel}", f"results/{rel}")
print("만든 파일:", zp, f"({os.path.getsize(zp)/1e6:.1f}MB, {len(files)}개 파일)")
print("빠진 파일:", missing if missing else "없음")

## 6. 저장 마무리 후 런타임 반납
보낼 zip이 만들어졌고 빠진 파일이 없으면 드라이브 저장을 마무리하고 런타임을 반납한다(컴퓨팅 단위 절약).

In [ ]:
ner_lib.verify_saved_models()   # 이번에 다룬 실험의 최종 가중치가 하나라도 없으면 여기서 멈추고 반납하지 않음
print("가중치 위치:", MODELS)
if os.path.exists(zp) and not missing:
    print("저장을 마무리하고 런타임을 반납합니다. 보낼 파일:", zp)
    drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()
else:
    print("빠진 파일이 있어 반납하지 않습니다:", missing)